# 06 · Extensões: faixas calibradas (T1), reserva por custo (T2), incerteza sem dados (T2) e escala × rotatividade (T3)

Quatro melhorias sobre os notebooks 02 a 05, cada uma validada e com resultado salvo em `resultados/`:

1. **T1:** encolhimento bayesiano empírico entre empresas e **faixas calibradas fora da amostra** (a faixa do notebook 02 era calibrada com todas as origens, inclusive as futuras).
2. **T2:** a reserva para ausência curta passa a ser escolhida pelo **menor custo esperado**, considerando que a própria reserva também falta, em vez de uma regra fixa de 95%.
3. **T2:** o erro de extrapolação medido (RMSE de 5,4 p.p.) é **propagado** para as células sem dado próprio (empresa 5 e ônibus da empresa 8).
4. **T3:** estimativa da **associação entre a escala praticada e a taxa de desligamento** (modelo de Poisson com exposição).

In [1]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rotatividade as rot, rotatividade_eb as reb, reserva_custo as rc
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
integrantes = rot.carregar_integrantes('../dados/dim_integrante.csv')
serie = rot.serie_mensal(integrantes)
S = {e: d.sort_values('mes').reset_index(drop=True) for e, d in serie.groupby('empresa')}
proj = pd.read_csv('../resultados/trilha1_projecao.csv', encoding='utf-8-sig').set_index('empresa')

## 1. T1 · Encolhimento entre empresas: o ponto central muda?

In [2]:
linhas = []
for H in (12, 24, 36):
    bt = reb.backtest(S, H)
    linhas.append({'H (meses)': H, 'origens': len(bt), 'MAE média 36m': (bt.media36 - bt.real).abs().mean(),
                   'MAE encolhida': (bt.eb36 - bt.real).abs().mean(),
                   'MAE emp. 3 (média)': (bt[bt.empresa == 3].media36 - bt[bt.empresa == 3].real).abs().mean(),
                   'MAE emp. 3 (encolhida)': (bt[bt.empresa == 3].eb36 - bt[bt.empresa == 3].real).abs().mean()})
pd.DataFrame(linhas).set_index('H (meses)').round(2)

,origens,MAE média 36m,MAE encolhida,MAE emp. 3 (média),MAE emp. 3 (encolhida)
H (meses),,,,,
12,303,6.33,6.27,10.68,9.27
24,243,7.38,7.48,7.35,6.34
36,184,9.81,9.85,NaN,NaN


O encolhimento empata com a média de 36 meses no conjunto e só ajuda a empresa 3 (a única pequena com histórico no backtest). **O ponto central do notebook 02 se mantém**; o encolhimento confirma que a escolha não é frágil.

## 2. T1 · As faixas cobrem o que dizem cobrir?

In [3]:
cob = []
for H in (12, 24):
    bt = reb.backtest(S, H)
    antiga = reb.cobertura_faixa(bt)                       # P10-P90 por empresa, calibrada como no notebook 02
    nova = reb.cobertura_lognormal(bt, z=1.2816)           # log-normal encolhida, só com o passado
    cob.append({'H (meses)': H, 'nominal': '80%', 'antiga, dentro da amostra': antiga.dentro.mean() * 100,
                'antiga, fora da amostra': antiga.fora.mean() * 100, 'nova, fora da amostra': nova.fora.mean() * 100})
pd.DataFrame(cob).set_index('H (meses)').round(0)

,nominal,"antiga, dentro da amostra","antiga, fora da amostra","nova, fora da amostra"
H (meses),,,,
12,80%,79.0,54.0,83.0
24,80%,79.0,37.0,69.0


A faixa do notebook 02 parece cobrir 80% porque foi calibrada com as mesmas origens que avalia. **Sem olhar o futuro, cobre só 54% (12 meses) e 37% (24 meses).** A faixa nova (razão realizado/projetado em escala log, com centro e dispersão de cada empresa encolhidos para os do grupo) chega a 83% e ~70%. Com 36 meses há poucas origens independentes e a calibração não é confiável (na empresa 6, a quebra de regime de 2020–2023 produz razões de até 5×), por isso usamos a faixa **anual**, limitada pelo maior turnover de 12 meses já observado.

In [4]:
def teto(e):
    d = S[e]; t = ((d.adm + d.resc).rolling(12).sum() / 2) / d.hc.rolling(12).mean() * 100
    return t[d.hc.rolling(12).mean() >= 20].max()

bt12 = reb.backtest(S, 12); bt12['razao'] = bt12.real / bt12.media36
eb, _, _ = reb.taxas_eb(S, pd.Timestamp('2026-09-01'))
fx = []
for e in (5, 8, 6, 3, 7, 4):
    lo, c, hi = reb.faixa_lognormal(bt12[bt12.empresa == e].razao, bt12.razao, z=1.2816)
    base = proj.loc[e, 'turnover_base_%']; T = teto(e) if e != 4 else max(teto(3), teto(7))
    fx.append({'empresa': e, 'base (média 36m)': base, 'encolhida': eb.get(e, np.nan), 'faixa nova: P10': base * lo,
               'faixa nova: P90 (limitada ao teto)': min(base * hi, max(T, base)), 'teto histórico 12m': T,
               'faixa antiga': f"{proj.loc[e, 'contrato_baixo_%']:.1f} a {proj.loc[e, 'contrato_alto_%']:.1f}"})
fx = pd.DataFrame(fx).set_index('empresa')
fx.round(2).to_csv('../resultados/trilha1_faixa_calibrada.csv', encoding='utf-8-sig')
fx.round(1)

,base (média 36m),encolhida,faixa nova: P10,faixa nova: P90 (limitada ao teto),teto histórico 12m,faixa antiga
empresa,,,,,,
5,10.2,10.2,7.6,12.8,12.8,10.0 a 12.8
8,28.5,28.5,21.7,34.5,34.5,28.5 a 34.5
6,61.1,61.0,35.8,68.9,68.9,61.1 a 71.0
3,32.8,32.5,15.3,46.4,46.4,20.6 a 41.1
7,13.6,14.2,6.4,24.5,24.5,13.6 a 24.5
4,27.0,49.7,14.2,46.4,46.4,20.4 a 46.4


O limite de cima quase não muda (já era o teto histórico). **O de baixo desce abaixo da base**: a regra "a base é o piso" do notebook 02 vinha da calibração dentro da amostra e não se sustenta fora dela. Para precificar, a leitura prudente continua sendo a parte alta da faixa, mas a faixa honesta é mais larga.

## 3. T2 · Reserva para ausência curta escolhida por custo

Num dia com taxa de ausência curta p, escalam-se Q titulares + R reservas, e cada um falta com probabilidade p (a reserva também falta). Presentes: S ~ Binomial(Q + R, 1 − p); postos descobertos: U = max(0, Q − S). A distribuição de p é a **empírica** dos 551 dias observados (que já contém o choque comum), reescalada para a média de cada empresa. O custo diário, em pessoa-dia, é **R + k·E[U]**, em que k é o custo de um posto descoberto: ~1,5 se for coberto com hora extra (50%), ~3 se houver custo operacional adicional, ~10 se gerar multa contratual ou viagem não realizada.

In [5]:
dia = pd.read_csv('../resultados/trilha2_ausencia_curta_diaria.csv', encoding='utf-8-sig')
dec = pd.read_csv('../resultados/trilha2_decomposicao_por_empresa.csv', encoding='utf-8-sig').set_index('empresa')
p0 = dia.p.values
res = []
for e, r in dec.iterrows():
    perda = r['perda_total_%'] / 100; Q = int(round(r.ativos * (1 - perda)))
    mu = r.curta_media / 100; ps = p0 * mu / p0.mean()
    a_plan = (r.ferias + r.prolongada) / 100
    folha_regra = Q * (1 + r['extra_%_base'] / 100)                 # regra de 95% (notebook 03)
    R_regra = int(round(folha_regra * (1 - a_plan) - Q))
    av = rc.avaliar(Q, R_regra, ps)
    res.append({'empresa': e, 'posições Q': Q, 'k': 'regra 95%', 'reserva R': R_regra, 'R/Q %': R_regra / Q * 100,
                'atendimento %': av['atendimento'] * 100, 'P(algum descoberto)': av['P_algum_descoberto'],
                'quadro extra %': r['extra_%_base']})
    for k in (1.5, 3, 10):
        o = rc.reserva_otima(Q, ps, k, R0=max(0, int(Q * mu * .4)))
        res.append({'empresa': e, 'posições Q': Q, 'k': k, 'reserva R': o['R'], 'R/Q %': o['R'] / Q * 100,
                    'atendimento %': o['atendimento'] * 100, 'P(algum descoberto)': o['P_algum_descoberto'],
                    'quadro extra %': rc.quadro_extra_pct(Q, o['R'], a_plan)})
res = pd.DataFrame(res)
res.round(3).to_csv('../resultados/trilha2_reserva_por_custo.csv', index=False, encoding='utf-8-sig')
res.pivot(index='empresa', columns='k', values='quadro extra %').round(1)

k,1.5,3,10,regra 95%
empresa,,,,
3,10.2,11.3,13.6,16.0
4,9.7,9.7,31.6,30.5
5,16.6,17.7,19.3,21.6
6,13.6,14.8,16.3,18.4
7,15.9,22.5,29.2,33.4
8,14.8,16.0,17.7,19.9


In [6]:
res.set_index(['empresa', 'k'])[['posições Q', 'reserva R', 'R/Q %', 'atendimento %', 'P(algum descoberto)']].round(3)

posições Q  reserva R   R/Q %  atendimento %  P(algum descoberto)
empresa k                                                                           
3       regra 95%          97          7   7.216         99.892                0.036
        1.5                97          2   2.062         98.751                0.494
        3                  97          3   3.093         99.242                0.310
        10                 97          5   5.155         99.731                0.105
4       regra 95%           5          1  20.000         99.638                0.017
        1.5                 5          0   0.000         96.758                0.150
        3                   5          0   0.000         96.758                0.150
        10                  5          1  20.000         99.638                0.017
5       regra 95%        2321        163   7.023         99.895                0.032
        1.5              2321         60   2.585         99.025                0.683
        3                2321         83   3.576         99.514                0.344
        10               2321        116   4.998         99.786                0.104
6       regra 95%         642         42   6.542         99.914                0.030
        1.5               642         14   2.181         99.030                0.663
        3                 642         21   3.271         99.559                0.311
        10                642         30   4.673         99.817                0.094
7       regra 95%          34          6  17.647         99.604                0.051
        1.5                34          1   2.941         95.303                0.661
        3                  34          3   8.824         98.271                0.265
        10                 34          5  14.706         99.379                0.087
8       regra 95%         816         57   6.985         99.896                0.033
        1.5               816         20   2.451         98.954                0.676
        3                 816         29   3.554         99.500                0.333
        10                816         41   5.025         99.787                0.103

**Leitura.** A regra de 95% equivale a supor que um posto descoberto custa mais de 10 pessoas-dia. Se a operação cobre o descoberto com hora extra (k ≈ 1,5), a reserva ótima é bem menor e o quadro extra cai alguns pontos percentuais em todas as empresas, aceitando ~1% de postos-dia cobertos por hora extra. Se o contrato pune o descoberto (k ≈ 10), a reserva ótima fica próxima da regra. **O dado que decide é o custo de um posto descoberto no contrato**, e por isso o mostramos como parâmetro.

## 4. T2 · Incerteza das células sem dado próprio

A validação do notebook 03 mediu o erro de extrapolar a perda para células sem dados: RMSE de 5,4 p.p. Propagamos esse erro (±1 RMSE na perda) para as células com peso de dados zero.

In [7]:
ext = []
for e, r in dec.iterrows():
    perda = r['perda_total_%'] / 100
    sem_dado = {5: 1.0, 8: 746 / 978}.get(e, 0.0)           # parcela do quadro sem dado próprio (notebook 03)
    for nome, d in (('-1 RMSE', -.054), ('base', 0), ('+1 RMSE', .054)):
        p = perda + sem_dado * d
        ext.append({'empresa': e, 'cenário': nome, 'quadro extra %': p / (1 - p) * 100})
ext = pd.DataFrame(ext).pivot(index='empresa', columns='cenário', values='quadro extra %')[['-1 RMSE', 'base', '+1 RMSE']]
ext.round(2).to_csv('../resultados/trilha2_faixa_extrapolacao.csv', encoding='utf-8-sig')
ext.round(1)

cenário,-1 RMSE,base,+1 RMSE
empresa,,,
3,16.0,16.0,16.0
4,30.5,30.5,30.5
5,14.1,21.6,30.2
6,18.4,18.4,18.4
7,33.4,33.4,33.4
8,14.2,19.9,26.1


Na empresa 5 a faixa honesta é de ~14% a ~30% das posições, bem mais larga que a faixa baixo–alto do notebook 03 (19% a 24%), que só refletia a escolha da referência. **É o maior risco do preço, e só se reduz com o ponto da empresa 5.**

## 5. T3 · A escala praticada está associada à rotatividade?

In [8]:
%run ../src/efeito_escala_rotatividade.py ../dados

vínculos: 1561
           n  saidas  voluntarias    meses
escala                                    
5x2      447     160           23   5619.8
6x1     1114     421          215  11570.3
saiu  | todas as empresas  | RR 6x1/5x2 = 0.81  IC95% [0.63, 1.04]  p = 0.094  n = 1561
saiu  | só empresa 6       | RR 6x1/5x2 = 1.16  IC95% [0.71, 1.90]  p = 0.558  n = 1056
vol   | todas as empresas  | RR 6x1/5x2 = 1.74  IC95% [1.00, 3.04]  p = 0.052  n = 1561
vol   | só empresa 6       | RR 6x1/5x2 = 1.75  IC95% [0.80, 3.84]  p = 0.162  n = 1056


Controlando empresa, grupo de função e tempo de casa, o total de saídas não difere entre 6x1 e 5x2, mas as **saídas voluntárias são 1,74 vez mais frequentes no 6x1** (IC95% 1,00 a 3,04; p = 0,052). É associação, não efeito causal; aponta que o 5x2 pode reduzir as saídas voluntárias (~43% a menos se o excesso desaparecer), o que na empresa 6 equivale a ~20% a menos de turnover.